<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/RAG-base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
!pip install pypdf faiss-cpu sentence-transformers google-generativeai -q

In [71]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print(pdf_path)

Saving User_Churn_Report_Metric_Definitions_v3.pdf to User_Churn_Report_Metric_Definitions_v3 (3).pdf
User_Churn_Report_Metric_Definitions_v3 (3).pdf


In [72]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print(text[:1000])


User Churn Report
Metric Glossary & Definitions
Version 3 | Updated: October 2026 | Internal Reference Only
Change indicators:
 UPDATED
Definition updated
 RENAMED
Metric renamed
 NEW
New metric added
Panel Size & Acquisition
Total Panelists
Joined
 UPDATED
Number of panelists who joined the panel in the given month.
■ Previously 'Total Users' (cumulative). Now shows monthly new joins only.
DOI (Double Opt-In)
Count of newly joined panelists in the month whose email address has been
verified.
Joined & Attempted
Count of panelists who joined in a given month and made at least one survey
attempt within that same join month.
Survey Start Rate
Share of newly verified panelists who went on to attempt at least one survey in
their join month.
 Formula: Joined & Attempted / DOI x 100
Activity & Engagement
Total Active
(Monthly)
 RENAMED
Number of panelists who participated in at least one survey in both the current
month AND the immediately preceding month (consecutive month activity). A
panel

In [73]:
chunk_size = 500

chunks = [
    text[i:i+chunk_size]
    for i in range(0, len(text), chunk_size)
]

print("Total Chunks:", len(chunks))


Total Chunks: 8


In [74]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(8, 384)


In [75]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors Added:", index.ntotal)

Vectors Added: 8


In [76]:
query = "How to calculate active last month not his month?"

In [77]:
query_embedding = embedding_model.encode([query])

distances, indices = index.search(
    np.array(query_embedding),
    k=3
)

retrieved_chunks = [chunks[i] for i in indices[0]]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\nChunk {i+1}\n")
    print(chunk[:500])


Chunk 1

ist is counted only if they were active in two back-to-back months.
■ Previously 'Total Active'. Now explicitly labelled Monthly and requires consecutive-month
activity.
Total Inactive
 UPDATED
Total number of panelists in the panel minus the number of panelists who were
active in the given month.
 Formula: Total Panel Size - Total Active (Monthly)
■ Previously 'Inactive'. Now 'Total Inactive' with a panel-wide denominator.

Active Last Month, Not
This Month
Count of panelists who attempted at l

Chunk 2

east one survey last month but have not
attempted any survey in the current month. Forms the basis of the Churn Rate
calculation.
Retention & Churn
Churn Rate
 UPDATED
Of the panelists active last month, the percentage who showed no survey activity
this month.
 Formula: (Active last month, not this month / Active last month) x 100
■ Denominator updated: now uses 'Active last month' (not total panel size).
Reactivated Users
Panelists who are active this month, were not active

In [78]:
!pip install langchain-text-splitters -q

In [79]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure a production-grade chunk splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Max size of each chunk (characters)
    chunk_overlap=50,     # Overlapping characters between adjacent chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""] # Attempts to split on paragraphs first, then sentences, etc.
)

# Split your document
production_chunks = text_splitter.split_text(text)

print(f"Total production chunks: {len(production_chunks)}")
print("\n--- First Chunk Preview ---\n")
print(production_chunks[0])

Total production chunks: 10

--- First Chunk Preview ---

User Churn Report
Metric Glossary & Definitions
Version 3 | Updated: October 2026 | Internal Reference Only
Change indicators:
 UPDATED
Definition updated
 RENAMED
Metric renamed
 NEW
New metric added
Panel Size & Acquisition
Total Panelists
Joined
 UPDATED
Number of panelists who joined the panel in the given month.
■ Previously 'Total Users' (cumulative). Now shows monthly new joins only.
DOI (Double Opt-In)
Count of newly joined panelists in the month whose email address has been
verified.


In [80]:
import google.generativeai as genai
from google.colab import userdata

# Retrieve the API key securely from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

genai.configure(
    api_key=api_key
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [81]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Great first,(Say:Hi 0r  Hello)
Answer only from the provided context i natural language.

Context:
{context}

Question:
{query}

If answer is not present,
say "Not found in document".
"""

In [82]:
# response = model.generate_content(
#     prompt
# )

# print(response.text)


In [83]:
# 1. Embed the production-grade chunks we generated using the Recursive Splitter
production_embeddings = embedding_model.encode(production_chunks)

# 2. Build a new FAISS index with these production chunks
prod_dimension = production_embeddings.shape[1]
prod_index = faiss.IndexFlatL2(prod_dimension)
prod_index.add(np.array(production_embeddings))

# 3. Embed and search for the query using the updated index
query_embedding = embedding_model.encode([query])
distances, indices = prod_index.search(np.array(query_embedding), k=3)

# Retrieve the best matching production chunks
retrieved_prod_chunks = [production_chunks[i] for i in indices[0]]
context_str = "\n\n".join(retrieved_prod_chunks)

# 4. Formulate an explicit prompt to guarantee the greeting style and natural language rules
prompt = f"""You are a helpful assistant.

CRITICAL INSTRUCTIONS:
1. You MUST begin your response with a friendly greeting, such as "Hi" or "Hello".
2. Answer the user's question using only the provided context in natural language.
3. If the answer is not present in the context, say "Not found in document".

Context:
{context_str}

Question:
{query}
"""

# 5. Generate content with the configured Gemini model
response = model.generate_content(prompt)
print(response.text)

Hello!
To calculate "Active Last Month, Not This Month", you need to count the panelists who attempted at least one survey last month but have not attempted any survey in the current month.
